# Frequency-Stratified Integrated Gradients

This notebook implements a frequency-aware explainability method by running Integrated Gradients (IG)
with **blurred baselines at increasing σ levels**. By differencing attributions between consecutive
blur levels, we isolate *which spatial frequency bands* the model relied on to make its prediction.

**Pipeline:**
1. Load a pretrained model + image
2. Generate baselines: blurred versions of the image at σ = [0, 2, 4, 8, 16, 32]
3. For each σ, compute IG attribution (integral of gradients from blurred → original)
4. Differential attributions: `ΔAttr(band_i) = Attr(σᵢ) - Attr(σᵢ₊₁)` → captures contribution of the removed frequency band
5. Visualise all bands + aggregate in a single inferno heatmap

In [ ]:
# ── Dependencies ─────────────────────────────────────────────────────────────
# pip install torch torchvision Pillow matplotlib scipy numpy

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import Normalize

import torch
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.models as models

from PIL import Image
from scipy.ndimage import gaussian_filter

print('All imports OK')

## 1 — Configuration

In [ ]:
# ── USER CONFIG ───────────────────────────────────────────────────────────────
import os

DATA_DIR   = '/home/arin_weling/wavex/wavelet_explanation/data'
OUTPUT_DIR = '/home/arin_weling/wavex/notebooks/outputs/freq_ig_fourier_explanation'
os.makedirs(OUTPUT_DIR, exist_ok=True)

IMAGE_NAMES = ['husky_lion','shih_tzu', 'husky_cat', 'cobra', 'banded_gecko', 'peacock', 'rock_python']

# Number of frequency bands per image.
N_BANDS  = 5

# Cutoff strategy:
#   'adaptive'  -> equal cumulative FFT-energy bands (image-specific)
#   'geometric' -> log-spaced radii between GEOM_MIN_CUTOFF and max_r
#   'octave'    -> powers-of-two style spacing (1, 2, 4, 8, ...)
CUTOFF_MODE = 'geometric'
GEOM_MIN_CUTOFF = 5

IG_STEPS = 100
DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')
print(f'Images to process: {IMAGE_NAMES}')
print(f'Output directory: {OUTPUT_DIR}')
print(f'Cutoff mode: {CUTOFF_MODE} (N_BANDS={N_BANDS}, GEOM_MIN_CUTOFF={GEOM_MIN_CUTOFF})')


## 2 — Load Model

In [ ]:
def load_model(device):
    """Load a pretrained ResNet-18. Swap for any torchvision model."""
    # model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model.eval()
    model.to(device)
    return model

model = load_model(DEVICE)
print('Model loaded — ResNet-18 (ImageNet)')

## 3 — Image Loading & Preprocessing

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

preprocess = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

def load_image(path):
    """Load image and return (PIL image, normalised tensor [1,3,H,W])."""
    pil_img = Image.open(path).convert('RGB')
    pil_img = pil_img.resize((224, 224), Image.BILINEAR)
    tensor  = preprocess(pil_img).unsqueeze(0)
    return pil_img, tensor

print('Preprocessing pipeline defined.')

## 4 — Determine Target Class

In [ ]:
# ── Set this to explore a single image ────────────────────────────────────────
IMAGE_PATH = os.path.join(DATA_DIR, 'husky_cat.jpg')   # <-- change as needed
TARGET_CLASS = None  # None = use top-1 prediction

# ── Load & display ─────────────────────────────────────────────────────────────
pil_img, img_tensor = load_image(IMAGE_PATH)
img_tensor = img_tensor.to(DEVICE)
print(f'Image tensor shape: {img_tensor.shape}')

plt.figure(figsize=(4, 4))
plt.imshow(pil_img)
plt.axis('off')
plt.title('Input image')
plt.tight_layout()
plt.show()

# ── Determine target class ─────────────────────────────────────────────────────
with torch.no_grad():
    logits = model(img_tensor)
    probs  = F.softmax(logits, dim=1)

top5_prob, top5_idx = probs.topk(5, dim=1)
print('Top-5 predictions:')
for i, (p, idx) in enumerate(zip(top5_prob[0], top5_idx[0])):
    print(f'  {i+1}. class {idx.item():>4d}  ({p.item()*100:.2f}%)')

if TARGET_CLASS is None:
    TARGET_CLASS = top5_idx[0, 0].item()
print(f'\nUsing target class: {TARGET_CLASS}')

## 5 — Fourier Low-Pass Baselines

For each cutoff radius `r` we apply a **square low-pass filter in Fourier space** (before normalisation) and then re-normalise.

The filter keeps all FFT coefficients `(u, v)` with `|u| ≤ r` AND `|v| ≤ r` (a square centred on DC) and zeros out the rest.

- `r = 0`   → only the DC term survives → flat mean-colour image (maximally blurred)
- `r = 112` → full spectrum retained → original image unchanged

This is the Fourier equivalent of Gaussian blur: a smaller square = stronger low-pass = blurrier baseline.

In [ ]:
# Cache normalization tensors per device to avoid repeated allocation
_NORM_CACHE = {}

def _get_norm_tensors(device):
    """Get cached mean/std tensors for the given device."""
    if device not in _NORM_CACHE:
        _NORM_CACHE[device] = (
            torch.tensor(IMAGENET_MEAN, device=device, dtype=torch.float32).view(1, 3, 1, 1),
            torch.tensor(IMAGENET_STD,  device=device, dtype=torch.float32).view(1, 3, 1, 1)
        )
    return _NORM_CACHE[device]


def load_image(path):
    """Load image and return (PIL image, normalised tensor [1,3,H,W])."""
    pil_img = Image.open(path).convert('RGB')
    pil_img = pil_img.resize((224, 224), Image.BILINEAR)
    tensor  = preprocess(pil_img).unsqueeze(0)
    return pil_img, tensor


def lowpass_tensor(tensor, cutoff):
    """
    Square low-pass filter a normalised image tensor [1,3,H,W] via FFT.

    Steps:
      1. Un-normalise to pixel space [0, 1]
      2. FFT2 each channel
      3. Keep only coefficients (u,v) with |u| <= cutoff AND |v| <= cutoff
         (a square of side 2*cutoff+1 centred on DC)
      4. IFFT2 back to pixel space
      5. Re-normalise with ImageNet stats

    cutoff=0  -> DC only -> uniform mean-colour image
    cutoff>=H//2 -> all frequencies retained -> original image
    """
    H, W = tensor.shape[-2], tensor.shape[-1]
    if cutoff >= H // 2:
        return tensor.clone()

    mean, std = _get_norm_tensors(tensor.device)

    # Un-normalise to [0, 1]
    pixel = (tensor * std + mean).squeeze(0)   # (3, H, W)

    # FFT with DC shifted to centre
    fft_shifted = torch.fft.fftshift(
        torch.fft.fft2(pixel), dim=(-2, -1)
    )  # (3, H, W) complex

    # Build square mask centred at (cy, cx)
    cy, cx = H // 2, W // 2
    mask = torch.zeros(H, W, dtype=torch.bool, device=tensor.device)
    mask[max(0, cy - cutoff): cy + cutoff + 1,
         max(0, cx - cutoff): cx + cutoff + 1] = True

    # Zero out everything outside the square
    fft_filtered = fft_shifted * mask.unsqueeze(0)   # broadcast over channels

    # IFFT back to pixel space
    filtered = torch.fft.ifft2(
        torch.fft.ifftshift(fft_filtered, dim=(-2, -1))
    ).real.clamp(0.0, 1.0).unsqueeze(0)             # (1, 3, H, W)

    # Re-normalise
    return (filtered - mean) / std


def _clean_cutoffs(cutoffs, max_r):
    """Clamp, deduplicate, and sort cutoff radii; always include max_r as the last point."""
    clean = []
    for c in cutoffs:
        ci = int(c)
        if 0 <= ci <= max_r:
            clean.append(ci)
    clean.append(max_r)   # always include full-spectrum endpoint; do NOT force 0
    return sorted(set(clean))


def compute_adaptive_cutoffs(img_tensor, n_bands=5, min_r=1):
    """
    Compute frequency cutoff radii from the image's cumulative energy distribution.

    Finds the radii at which cumulative FFT energy reaches evenly-spaced percentiles
    [0%, 1/n, 2/n, ..., 100%], so each band captures an equal fraction of the
    image's total frequency energy.
    """
    mean, std = _get_norm_tensors(img_tensor.device)

    # Un-normalise to pixel space
    img_px = (img_tensor * std + mean).squeeze(0)   # (3, H, W)

    # FFT energy spectrum (sum over RGB channels), with DC centered to match radius grid
    fft_shifted = torch.fft.fftshift(torch.fft.fft2(img_px), dim=(-2, -1))
    energy = (fft_shifted.abs() ** 2).sum(dim=0)   # (H, W)

    # Chebyshev distance from DC for each coefficient
    H, W = energy.shape
    cy, cx = H // 2, W // 2
    max_r = min(H, W) // 2

    y = torch.arange(H, device=img_tensor.device).view(-1, 1) - cy
    x = torch.arange(W, device=img_tensor.device).view(1, -1) - cx
    radius_grid = torch.maximum(y.abs(), x.abs())   # (H, W) square metric

    # Cumulative energy: how much energy is in the square of radius <= r?
    cum_energy = torch.zeros(max_r + 1, device=img_tensor.device)
    for r in range(max_r + 1):
        cum_energy[r] = energy[radius_grid <= r].sum()

    cum_norm = cum_energy / cum_energy[-1]   # normalised to [0, 1]

    # Find radii at evenly-spaced percentiles — start from min_r
    cutoffs = [max(0, int(min_r))]
    for i in range(1, n_bands):
        target = i / n_bands
        idx = int((cum_norm - target).abs().argmin().item())
        if idx not in cutoffs:
            cutoffs.append(idx)
    cutoffs.append(max_r)

    return _clean_cutoffs(cutoffs, max_r)


def compute_geometric_cutoffs(img_tensor, n_bands=5, min_r=1):
    """Compute log-spaced cutoff radii between min_r and max_r."""
    H, W = img_tensor.shape[-2], img_tensor.shape[-1]
    max_r = min(H, W) // 2
    min_r = max(1, min(int(min_r), max_r))

    # n_bands points from min_r to max_r — no forced r=0 prefix
    geom = np.geomspace(min_r, max_r, num=max(2, n_bands)).round().astype(int).tolist()
    return _clean_cutoffs(geom, max_r)


def compute_octave_cutoffs(img_tensor, n_bands=5, min_r=1):
    """Compute octave-like cutoff radii (powers-of-two progression)."""
    H, W = img_tensor.shape[-2], img_tensor.shape[-1]
    max_r = min(H, W) // 2
    min_r = max(1, min(int(min_r), max_r))

    # Start from min_r, not 0
    cutoffs = []
    r = min_r
    while r < max_r and len(cutoffs) < n_bands + 1:
        cutoffs.append(r)
        r *= 2
    cutoffs.append(max_r)

    # If image size is small and octave progression is too short, top up using geometric spacing.
    if len(set(cutoffs)) < n_bands + 1:
        geom = np.geomspace(min_r, max_r, num=max(2, n_bands)).round().astype(int).tolist()
        cutoffs.extend(geom)

    return _clean_cutoffs(cutoffs, max_r)


def compute_cutoffs(img_tensor, n_bands=5, mode='adaptive', geometric_min_r=1):
    """Dispatch cutoff computation by mode: adaptive, geometric, or octave."""
    mode = str(mode).strip().lower()

    if mode == 'adaptive':
        cutoffs = compute_adaptive_cutoffs(img_tensor, n_bands=n_bands, min_r=geometric_min_r)
    elif mode == 'geometric':
        cutoffs = compute_geometric_cutoffs(
            img_tensor, n_bands=n_bands, min_r=geometric_min_r
        )
    elif mode == 'octave':
        cutoffs = compute_octave_cutoffs(
            img_tensor, n_bands=n_bands, min_r=geometric_min_r
        )
    else:
        raise ValueError("CUTOFF_MODE must be one of: 'adaptive', 'geometric', 'octave'")

    if len(cutoffs) < 2:
        raise ValueError(f'Need at least 2 cutoffs, got {cutoffs}')
    return cutoffs


print('Fourier low-pass, cutoff strategies, and IG functions defined.')

In [ ]:
# ── Visualise Fourier Low-Pass Baselines (configurable cutoffs) ───────────────
# Uses img_tensor from the 'Determine Target Class' cell above.

mean_np = np.array(IMAGENET_MEAN).reshape(3, 1, 1)
std_np  = np.array(IMAGENET_STD ).reshape(3, 1, 1)

def tensor_to_rgb(t):
    return (t.squeeze(0).cpu().numpy() * std_np + mean_np).clip(0, 1).transpose(1, 2, 0)

# Compute cutoffs for the current image using selected strategy
cutoff_sizes = compute_cutoffs(
    img_tensor,
    n_bands=N_BANDS,
    mode=CUTOFF_MODE,
    geometric_min_r=GEOM_MIN_CUTOFF,
)
print(f'Cutoff mode: {CUTOFF_MODE}')
print(f'Cutoff radii: {cutoff_sizes}')

n_cols = 1 + len(cutoff_sizes)
fig, axes = plt.subplots(1, n_cols, figsize=(3.2 * n_cols, 3.5), facecolor='#0d0d0d')

axes[0].imshow(tensor_to_rgb(img_tensor))
axes[0].set_title('Original', color='white', fontsize=10, pad=6)
axes[0].axis('off')

for i, r in enumerate(cutoff_sizes):
    baseline = lowpass_tensor(img_tensor, r)
    axes[i + 1].imshow(tensor_to_rgb(baseline))
    label = f'r={r}\n({2*r+1}x{2*r+1})' if r > 0 else 'r=0\n(DC only)'
    axes[i + 1].set_title(label, color='white', fontsize=9, pad=6)
    axes[i + 1].axis('off')

fig.suptitle(f'Fourier Low-Pass Baselines ({CUTOFF_MODE})', color='white', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()


## 6 — Integrated Gradients

For a given baseline `x'` we compute:

$$\text{IG}_i(x) = (x_i - x'_i) \times \int_{\alpha=0}^{1} \frac{\partial F(x' + \alpha(x - x'))}{\partial x_i} \, d\alpha$$

Approximated via the **Riemann sum** over `IG_STEPS` steps.

In [ ]:
def integrated_gradients(model, img, baseline, target_class, steps=50, batch_size=10):
    """
    Compute IG attributions with Fourier-space interpolation.

    Instead of blending pixel values directly, the interpolation path is defined
    in Fourier space:

        FFT_interp(α) = FFT(baseline) + α × (FFT(img) - FFT(baseline))
        interp(α)     = IFFT( FFT_interp(α) )

    At α=0 → baseline (low-pass filtered image)
    At α=1 → original image (full spectrum)

    Each step along the path is an image whose frequency square grows from the
    baseline cutoff up to the full spectrum — semantically meaningful for
    frequency attribution.

    Pixel-space gradients are still collected at each step:
        IG_i(x) = (x_i - x'_i) × (1/steps) × Σ ∂F(interp(α)) / ∂pixel_i

    Returns
    -------
    attribution : np.ndarray, shape (H, W)
        Absolute attribution magnitude (summed over channels).
    attribution_raw : np.ndarray, shape (3, H, W)
        Signed attribution per channel.
    """
    baseline = baseline.detach()
    img      = img.detach()

    mean, std = _get_norm_tensors(img.device)

    # ── Un-normalise to pixel space and take FFT ──────────────────────────────
    img_px      = (img      * std + mean).squeeze(0)   # (3, H, W)
    baseline_px = (baseline * std + mean).squeeze(0)   # (3, H, W)

    fft_img      = torch.fft.fftshift(torch.fft.fft2(img_px),      dim=(-2, -1))  # (3, H, W) complex
    fft_baseline = torch.fft.fftshift(torch.fft.fft2(baseline_px), dim=(-2, -1))  # (3, H, W) complex
    fft_delta    = fft_img - fft_baseline                                          # (3, H, W) complex

    # ── Pixel-space delta for final IG multiplication ─────────────────────────
    # IFFT is linear so IFFT(fft_delta) = img - baseline, same result either way
    delta = img - baseline   # (1, 3, H, W)

    alphas    = torch.linspace(0, 1, steps + 1, device=img.device)
    grads_all = []

    for i in range(0, len(alphas), batch_size):
        alpha_batch = alphas[i : i + batch_size]   # (B,)
        B = alpha_batch.shape[0]

        # ── Interpolate in Fourier space ──────────────────────────────────────
        # fft_baseline/fft_delta: (3,H,W) → broadcast to (B,3,H,W)
        fft_interp = (fft_baseline.unsqueeze(0) +
                      alpha_batch.view(B, 1, 1, 1) * fft_delta.unsqueeze(0))   # (B, 3, H, W) complex

        # ── IFFT back to pixel space and re-normalise ─────────────────────────
        px_interp = torch.fft.ifft2(
            torch.fft.ifftshift(fft_interp, dim=(-2, -1))
        ).real.clamp(0.0, 1.0)               # (B, 3, H, W)

        interp = (px_interp - mean) / std    # (B, 3, H, W), normalised
        interp.requires_grad_(True)

        # ── Forward + backward ────────────────────────────────────────────────
        logits = model(interp)
        score  = logits[:, target_class].sum()
        score.backward()

        grads_all.append(interp.grad.detach())

    # ── Riemann sum ───────────────────────────────────────────────────────────
    grads     = torch.cat(grads_all, dim=0)          # (steps+1, 3, H, W)
    avg_grads = grads.mean(dim=0, keepdim=True)       # (1, 3, H, W)

    ig_raw = (delta * avg_grads).squeeze(0).cpu().numpy()   # (3, H, W)
    ig_abs = np.abs(ig_raw).sum(axis=0)                     # (H, W)

    return ig_abs, ig_raw


def normalise(arr, clip_percentile=99):
    """Normalize array to [0, 1] by clipping high outliers and scaling."""
    arr  = np.maximum(arr, 0)
    vmax = np.percentile(arr, clip_percentile)
    if vmax == 0:
        return arr
    return arr / vmax


print('Fourier-interpolated IG and normalise functions defined.')


## 7 — Compute IG for Every Sigma Level

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# MAIN LOOP: Process all images
# ════════════════════════════════════════════════════════════════════════════════

for image_name in IMAGE_NAMES:
    print(f"\n{'='*80}")
    print(f"Processing: {image_name}")
    print(f"{'='*80}")

    # ── Find image file ───────────────────────────────────────────────────────
    image_path = None
    for ext in ['.jpg', '.png', '.jpeg', '.webp']:
        candidate = os.path.join(DATA_DIR, image_name + ext)
        if os.path.exists(candidate):
            image_path = candidate
            break

    if image_path is None:
        print(f"Image not found: {image_name}")
        continue

    # ── Load image ────────────────────────────────────────────────────────────
    pil_img, img_tensor = load_image(image_path)
    img_tensor = img_tensor.to(DEVICE)

    # ── Determine target class ────────────────────────────────────────────────
    with torch.no_grad():
        logits = model(img_tensor)
        probs  = F.softmax(logits, dim=1)

    top5_prob, top5_idx = probs.topk(5, dim=1)
    target_class = top5_idx[0, 0].item()
    top1_prob    = top5_prob[0, 0].item()
    print(f"Top-1 class: {target_class} ({top1_prob*100:.2f}%)")

    # ── Compute cutoff radii from selected strategy ───────────────────────────
    cutoff_sizes = compute_cutoffs(
        img_tensor,
        n_bands=N_BANDS,
        mode=CUTOFF_MODE,
        geometric_min_r=GEOM_MIN_CUTOFF,
    )
    print(f"Cutoff mode: {CUTOFF_MODE}")
    print(f"Cutoffs: {cutoff_sizes}")

    # ── Generate Fourier low-pass baselines ───────────────────────────────────
    baselines = [lowpass_tensor(img_tensor, r) for r in cutoff_sizes]
    print(f"Created {len(baselines)} baselines")

    # ── Compute IG for each cutoff level ──────────────────────────────────────
    ig_results = []
    for i, (cutoff, baseline) in enumerate(zip(cutoff_sizes, baselines)):
        print(f"  [{i+1}/{len(cutoff_sizes)}] IG for cutoff r={cutoff}...", end=' ')
        attr_abs, attr_raw = integrated_gradients(
            model, img_tensor, baseline, target_class, steps=IG_STEPS
        )
        ig_results.append(attr_abs)
        print(f"max={attr_abs.max():.4f}")

    # ── Differential band attributions ───────────────────────────────────────
    band_attributions = []
    band_labels       = []

    for i in range(len(cutoff_sizes) - 1):
        diff  = ig_results[i] - ig_results[i + 1]
        band_attributions.append(diff)
        band_labels.append(f'r {cutoff_sizes[i]}->{cutoff_sizes[i+1]}')

    n_bands = len(band_attributions)
    print(f"Computed {n_bands} frequency bands")

    # ── Visualize ─────────────────────────────────────────────────────────────
    clamped_bands    = [np.maximum(b, 0) for b in band_attributions]
    band_stack       = np.stack(clamped_bands, axis=0)
    dominant_band    = np.argmax(band_stack, axis=0)
    max_contribution = np.max(band_stack, axis=0)

    total_panels = 1 + n_bands + 1
    fig, axes    = plt.subplots(1, total_panels, figsize=(3.2 * total_panels, 4),
                                facecolor='#0d0d0d')

    CMAP      = 'inferno'
    all_vals  = np.concatenate([np.maximum(b, 0).ravel() for b in band_attributions])
    vmax_glob = np.percentile(all_vals, 99)

    mean_np  = np.array(IMAGENET_MEAN).reshape(3, 1, 1)
    std_np   = np.array(IMAGENET_STD ).reshape(3, 1, 1)
    vis_orig = (img_tensor.squeeze(0).cpu().numpy() * std_np + mean_np).clip(0, 1).transpose(1, 2, 0)

    axes[0].imshow(vis_orig)
    axes[0].set_title('Original', color='white', fontsize=10, pad=6)
    axes[0].axis('off')

    for i, (band, label) in enumerate(zip(band_attributions, band_labels)):
        ax   = axes[i + 1]
        data = np.maximum(band, 0)
        im   = ax.imshow(data, cmap=CMAP, vmin=0, vmax=vmax_glob)
        ax.set_title(label, color='white', fontsize=9, pad=6)
        ax.axis('off')
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.ax.yaxis.set_tick_params(color='white', labelcolor='white', labelsize=6)
        cbar.outline.set_edgecolor('white')

    ax_dom = axes[-1]
    cmap_spectrum = matplotlib.colormaps.get_cmap('hsv')
    colors = [cmap_spectrum(1.0 - (i + 1) / n_bands) for i in range(n_bands)]

    dom_rgb = np.zeros((*dominant_band.shape, 3))
    for band_idx in range(n_bands):
        dom_rgb[dominant_band == band_idx] = colors[band_idx][:3]

    dom_rgb_bright = np.clip(dom_rgb * normalise(max_contribution)[..., None], 0, 1)
    ax_dom.imshow(dom_rgb_bright)
    ax_dom.set_title('Dominant Band', color='white', fontsize=10, pad=6, fontweight='bold')
    ax_dom.axis('off')

    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=colors[i][:3], label=band_labels[i])
                       for i in range(n_bands)]
    # ax_dom.legend(handles=legend_elements, loc='upper left', fontsize=7,
    #               bbox_to_anchor=(0, 1), framealpha=0.9, facecolor='#0d0d0d',
    #               edgecolor='white', frameon=True)

    fig.suptitle(
        f'Fourier-Stratified IG  |  {image_name}  |  class {target_class}  |  mode {CUTOFF_MODE}  |  cutoffs {cutoff_sizes}',
        color='white', fontsize=11, y=1.01
    )
    plt.tight_layout()

    output_filename = os.path.join(OUTPUT_DIR, f'freq_ig_fourier_explanation_{image_name}.png')
    plt.savefig(output_filename, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    print(f"Saved: {output_filename}")
    plt.close()

print(f"\n{'='*80}")
print(f"All {len(IMAGE_NAMES)} images processed and saved to {OUTPUT_DIR}")
print(f"{'='*80}")


## 8 — Differential Frequency Band Attributions

The attribution map at cutoff `rᵢ` captures everything the model uses that is *above* the `rᵢ` low-pass cutoff — i.e. spatial frequencies higher than what the `rᵢ`-square baseline retains.

So the **differential**:

$$\Delta\text{Attr}(\text{band}_i) = \text{Attr}(r_i) - \text{Attr}(r_{i+1})$$

isolates the contribution of the frequency content that was present in the baseline at `rᵢ` but absent at `rᵢ₊₁`.

Smaller `r` bands → high-frequency details;  larger `r` bands → coarse / low-frequency structure.

The square low-pass mask (rather than Gaussian blur) gives **hard frequency boundaries** in Fourier space: each band is exactly the annular ring `rᵢ < max(|u|, |v|) ≤ rᵢ₊₁`.

## 9 — Visualise: Frequency Band Attributions + Aggregate

Each panel shows **where in the image** the model used information in that frequency band.

The last panel is the **weighted aggregate** — a single importance map across all frequencies.

## 10 — Frequency Importance Bar Chart

A global scalar importance score per band — answers *'which frequency range mattered most overall?'*

---
## Interpretation Guide
| Band r range | Frequency content | What a high score means |
|---|---|---|
| Low r (e.g. 0→2) | Very high frequencies — fine edges, grain | Model is **texture-biased** |
| Mid r (e.g. 5→10) | Mid frequencies — object parts, contours | Model uses **shape features** |
| High r (e.g. 20→40) | Low frequencies — colour blobs, silhouette | Model relies on **global structure** |
**Tips:**
- If most attribution is in low-r bands → model is texture-biased (common in standard ImageNet CNNs)
- If attribution spreads into high-r bands → model is shape/structure-biased (common after Stylized-ImageNet training)
- Bright hotspots in a high-r panel but not low-r = the region matters for its **coarse shape**, not fine detail
- You can adjust `N_BANDS` for finer or coarser frequency analysis